# 03 — Model Training

This notebook trains and evaluates the classical ML models:
- **Logistic Regression** (simple baseline)
- **XGBoost** (strong ML baseline)

All models are trained on the `final` feature set first, then looped over all feature sets.

Results are saved to `outputs/results/metrics.csv`.  
Plots are saved to `outputs/figures/models/`.

The PyTorch FNN is handled separately in a later cell / notebook.

## 0 · Imports

In [1]:
import sys
sys.path.append('..')

# Self modules
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.models import split_data, get_X_y, train_logistic_regression, train_xgboost
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve
from config import FEATURE_SETS, FEATURE_CONFIG

## 1 · Load & Split

Split is done on the **raw** DataFrame before preprocessing.  
This prevents any data leakage from imputation or encoding statistics.

In [2]:
df_raw = load_data()

df_train_raw, df_test_raw = split_data(df_raw, test_seasons=[2022, 2023])

Cache found - loading data from: C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\data\cache\pbp_raw.parquet
[split_data] Train seasons : [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021)]
[split_data] Test  seasons : [np.int32(2022), np.int32(2023)]
[split_data] Train rows    : 286,892
[split_data] Test  rows    : 99,099


## 2 · Preprocess

Train and test sets are preprocessed **independently** using the `final` feature set.  
The full feature-set loop comes in Section 4.

In [3]:
ACTIVE_FEATURE_SET = "final"

df_train = preprocess(df_train_raw, feature_set=FEATURE_SETS[ACTIVE_FEATURE_SET], feature_config=FEATURE_CONFIG)
df_test  = preprocess(df_test_raw,  feature_set=FEATURE_SETS[ACTIVE_FEATURE_SET], feature_config=FEATURE_CONFIG)


                              PREPROCESSING REPORT                              

INPUT:
   Shape: (286892, 372)
   Missing: 38714249 NaNs (36.28%)

OUTPUT:
   Shape: (203362, 8)
   Features (X): 7 columns
   Target (y): 203362 samples
   Missing: 0 NaNs

SELECTED FEATURES IMPUTATION DETAILS:
   Feature Name                   | NaNs Before  | NaNs After  | Filled/Removed
   ----------------------------------------------------------------------
   shotgun                        | 0            | 0           | 0        | -                       
   down                           | 743          | 0           | 743      | dropped (<1% NaNs)      
   score_differential             | 0            | 0           | 0        | -                       
   ydstogo                        | 0            | 0           | 0        | -                       
   posteam_timeouts_remaining     | 0            | 0           | 0        | -                       
   quarter_seconds_remaining      | 0         

In [4]:
# Sanity check
print(f"Train NaNs : {df_train.isna().sum().sum()}")
print(f"Test  NaNs : {df_test.isna().sum().sum()}")
print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")
print(f"Target distribution (train):\n{df_train['target'].value_counts(normalize=True).round(3)}")

Train NaNs : 0
Test  NaNs : 0
Train shape: (203362, 8)
Test  shape: (70778, 8)
Target distribution (train):
target
1    0.591
0    0.409
Name: proportion, dtype: float64


In [5]:
X_train, y_train = get_X_y(df_train)
X_test,  y_test  = get_X_y(df_test)

## 3 · Train & Evaluate — `final` Feature Set

### 3.1 Logistic Regression

In [6]:
lr_model = train_logistic_regression(X_train, y_train)

metrics_lr = evaluate_model(
    model=lr_model,
    X_test=X_test,
    y_test=y_test,
    model_name="LogisticRegression",
    feature_set=ACTIVE_FEATURE_SET,
)

plot_confusion_matrix(lr_model, X_test, y_test, "LogisticRegression", ACTIVE_FEATURE_SET)

[train_logistic_regression] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] LogisticRegression | feature_set=final
  Accuracy  : 0.7055
  F1        : 0.6991
  ROC-AUC   : 0.7574
  Precision : 0.7033
  Recall    : 0.7055

[plot_confusion_matrix] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\cm_LogisticRegression_final.png


### 3.2 XGBoost

In [7]:
xgb_model = train_xgboost(X_train, y_train)

metrics_xgb = evaluate_model(
    model=xgb_model,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    feature_set=ACTIVE_FEATURE_SET,
)

plot_confusion_matrix(xgb_model, X_test, y_test, "XGBoost", ACTIVE_FEATURE_SET)

c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:16:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[train_xgboost] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] XGBoost | feature_set=final
  Accuracy  : 0.7133
  F1        : 0.7064
  ROC-AUC   : 0.7781
  Precision : 0.712
  Recall    : 0.7133

[plot_confusion_matrix] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\cm_XGBoost_final.png


### 3.3 ROC Curve — beide Modelle im Vergleich

In [8]:
plot_roc_curve(
    models={
        "LogisticRegression": lr_model,
        "XGBoost":            xgb_model,
    },
    X_test=X_test,
    y_test=y_test,
    feature_set=ACTIVE_FEATURE_SET,
)

[plot_roc_curve] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\roc_final.png


## 4 · Feature Set Loop

Train both models on **all four feature sets** and collect metrics.

Results are appended to `metrics.csv` — existing rows for the same
`(model, feature_set)` combination are automatically overwritten.

In [ ]:
for fs_name, fs_cols in FEATURE_SETS.items():

    print(f"\n{'='*55}")
    print(f" Feature set: {fs_name}  ({len(fs_cols)} features)")
    print(f"{'='*55}")

    # Preprocess
    _df_train = preprocess(df_train_raw, feature_set=fs_cols, feature_config=FEATURE_CONFIG)
    _df_test  = preprocess(df_test_raw,  feature_set=fs_cols, feature_config=FEATURE_CONFIG)

    _X_train, _y_train = get_X_y(_df_train)
    _X_test,  _y_test  = get_X_y(_df_test)


    _X_test = _X_test.reindex(columns=_X_train.columns, fill_value=0) 
    # Logistic Regression
    _lr  = train_logistic_regression(_X_train, _y_train)
    evaluate_model(_lr,  _X_test, _y_test, "LogisticRegression", fs_name)

    # XGBoost
    _xgb = train_xgboost(_X_train, _y_train)
    evaluate_model(_xgb, _X_test, _y_test, "XGBoost", fs_name)

    # ROC curve per feature set
    plot_roc_curve(
        models={"LogisticRegression": _lr, "XGBoost": _xgb},
        X_test=_X_test,
        y_test=_y_test,
        feature_set=fs_name,
    )

print("\nDone. All results saved to metrics.csv")


 Feature set: final  (7 features)

                              PREPROCESSING REPORT                              

INPUT:
   Shape: (286892, 372)
   Missing: 38714249 NaNs (36.28%)

OUTPUT:
   Shape: (203362, 8)
   Features (X): 7 columns
   Target (y): 203362 samples
   Missing: 0 NaNs

SELECTED FEATURES IMPUTATION DETAILS:
   Feature Name                   | NaNs Before  | NaNs After  | Filled/Removed
   ----------------------------------------------------------------------
   shotgun                        | 0            | 0           | 0        | -                       
   down                           | 743          | 0           | 743      | dropped (<1% NaNs)      
   score_differential             | 0            | 0           | 0        | -                       
   ydstogo                        | 0            | 0           | 0        | -                       
   posteam_timeouts_remaining     | 0            | 0           | 0        | -                       
   quarter_

c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:16:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[train_xgboost] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] XGBoost | feature_set=final
  Accuracy  : 0.7133
  F1        : 0.7064
  ROC-AUC   : 0.7781
  Precision : 0.712
  Recall    : 0.7133

[plot_roc_curve] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\roc_final.png

 Feature set: 005  (8 features)

                              PREPROCESSING REPORT                              

INPUT:
   Shape: (286892, 372)
   Missing: 38714249 NaNs (36.28%)

OUTPUT:
   Shape: (203362, 9)
   Features (X): 8 columns
   Target (y): 203362 samples
   Missing: 0 NaNs

SELECTED FEATURES IMPUTATION DETAILS:
   Feature Name                   | NaNs Before  | NaNs After  | Filled/Removed
   ----------------------------------------------------------------------
   wp                             | 0            | 0           | 0        | -

c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:16:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[train_xgboost] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] XGBoost | feature_set=005
  Accuracy  : 0.7181
  F1        : 0.7115
  ROC-AUC   : 0.7857
  Precision : 0.717
  Recall    : 0.7181

[plot_roc_curve] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\roc_005.png

 Feature set: 003  (13 features)

                              PREPROCESSING REPORT                              

INPUT:
   Shape: (286892, 372)
   Missing: 38714249 NaNs (36.28%)

OUTPUT:
   Shape: (203362, 14)
   Features (X): 13 columns
   Target (y): 203362 samples
   Missing: 0 NaNs

SELECTED FEATURES IMPUTATION DETAILS:
   Feature Name                   | NaNs Before  | NaNs After  | Filled/Removed
   ----------------------------------------------------------------------
   goal_to_go                     | 0            | 0           | 0        | - 

c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:16:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[train_xgboost] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] XGBoost | feature_set=003
  Accuracy  : 0.7212
  F1        : 0.717
  ROC-AUC   : 0.7911
  Precision : 0.719
  Recall    : 0.7212

[plot_roc_curve] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\roc_003.png

 Feature set: 000  (20 features)

                              PREPROCESSING REPORT                              

INPUT:
   Shape: (286892, 372)
   Missing: 38714249 NaNs (36.28%)

OUTPUT:
   Shape: (203362, 28)
   Features (X): 27 columns
   Target (y): 203362 samples
   Missing: 0 NaNs

SELECTED FEATURES IMPUTATION DETAILS:
   Feature Name                   | NaNs Before  | NaNs After  | Filled/Removed
   ----------------------------------------------------------------------
   surface                        | 0            | 0           | 0        | -  

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- surface_a_turf
Feature names seen at fit time, yet now missing:
- surface_grass 
